Imports & configuration

In [5]:
!pip install kneed
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

from kneed import KneeLocator
import warnings
warnings.filterwarnings("ignore")

In [6]:
!pip install kneed

Load TF-IDF

In [7]:
TFIDF_PATH = "/content/HDFS_tfidf_full.csv"   # BlockId + TF-IDF features
BLOCK_COL = "BlockId"

df = pd.read_csv(TFIDF_PATH)
df[BLOCK_COL] = df[BLOCK_COL].astype(str)

X = df.drop(columns=[BLOCK_COL]).values
block_ids = df[BLOCK_COL].values

Dimensionality reduction (critical for DBSCAN accuracy)

DBSCAN does not work well in raw TF-IDF space.
SVD is mandatory, not optional.

In [8]:
SVD_DIM = 50   # 50–200 is safe for TF-IDF
svd = TruncatedSVD(
    n_components=SVD_DIM,
    algorithm="randomized",
    n_iter=7,
    random_state=42
)

X_svd = svd.fit_transform(X)

print(f"Explained variance (SVD): {svd.explained_variance_ratio_.sum():.3f}")

Explained variance (SVD): 1.000


Scaling (Euclidean distance must be meaningful)

In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_svd)

eps selection via k-distance knee (no labels)

This is the correct, textbook DBSCAN approach.

In [10]:
MIN_SAMPLES = 10

nn = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric="euclidean")
nn.fit(X_scaled)

distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

x = np.arange(len(k_distances))
knee = KneeLocator(
    x,
    k_distances,
    curve="convex",
    direction="increasing"
)

eps = float(k_distances[knee.knee])
print("Selected eps:", eps)

Selected eps: 17.165526245810312


DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=eps,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    n_jobs=-1
)

cluster_labels = dbscan.fit_predict(X_scaled)

Anomaly definition (pure DBSCAN)

In [ ]:
pred_anomaly = (cluster_labels == -1).astype(int)

anomaly_rate = pred_anomaly.mean()
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

print("Clusters found:", n_clusters)
print("Noise points:", pred_anomaly.sum())
print("Unsupervised anomaly rate:", anomaly_rate)

Evaluation using label file (evaluation ONLY)

In [ ]:
LABEL_PATH = "/content/HDFS_labels.csv"   # BlockId + Label (0/1)

labels_df = pd.read_csv(LABEL_PATH)
labels_df[BLOCK_COL] = labels_df[BLOCK_COL].astype(str)

pred_df = pd.DataFrame({
    BLOCK_COL: block_ids,
    "pred_anomaly": pred_anomaly
})

eval_df = labels_df.merge(pred_df, on=BLOCK_COL, how="inner")

y_true = eval_df["Label"].astype(int).values
y_pred = eval_df["pred_anomaly"].values

Metrics (proper anomaly metrics)

In [1]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary",
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Confusion matrix:\n", cm)
print("Evaluated samples:", len(eval_df))

NameError: name 'precision_recall_fscore_support' is not defined